Externally validate the ActiNet model on the WISDM and Realworld datasets

In [ ]:
import pandas as pd
import numpy as np
import os
import subprocess
from joblib import Parallel, delayed
from tqdm import tqdm

from actinet.prepare import download_data, make_acc_df
from actinet.actinet import load_classifier
from actinet.models import RFActivityClassifier
from actinet.utils.model_config import MODEL_CONFIG
from actinet.utils.utils import custom_date_parser
from actinet.utils.plot_utils import generate_confusion_matrices

WINSEC = 30
ANNOLABEL = "Walmsley2020"
activity_labels = MODEL_CONFIG[ANNOLABEL]["labels"].keys()

Download validation dataset from:

https://zenodo.org/records/6574265#.YovCMi8w1qs

In [ ]:
url = "https://zenodo.org/records/6574265/files/data/UKBB/ssl_data/data.zip?download=1"
zip_path = "data/raw/ukbb_data.zip"
extract_path = "data/raw/ukbb"

download_data(url, zip_path, extract_path)

In [ ]:
make_acc_df("data/raw/ukbb/data/downstream/realworld_30hz_clean", "data/prepared/real_world.csv", 30)
make_acc_df("data/raw/ukbb/data/downstream/wisdm_30hz_clean", "data/prepared/wisdm.csv", 30)

In [ ]:
clf = load_classifier(
    classifier="walmsley",
    model_repo_path=None,
    force_download=False,
    verbose=False
    )

clf.verbose = False

In [ ]:
df_rw = pd.read_csv("data/prepared/real_world.csv", index_col="time", parse_dates=True)

df_rw["label"] = df_rw["label"].map({'sitting': 'sedentary', 
                                     'walking': 'moderate-vigorous', 
                                     'lying': 'sedentary', 
                                     'climbingdown': 'moderate-vigorous', 
                                     'climbingup': 'moderate-vigorous',
                                     'standing': 'light', 
                                     'jumping': 'moderate-vigorous', 
                                     'running': 'moderate-vigorous'})

In [ ]:
os.makedirs(f"outputs/predictions/actinet/real_world", exist_ok=True)

def get_rw_predictions(pid):
    data = df_rw.loc[df_rw["pid"] == pid]
    y = clf.predict_from_frame(data, 30, sleep_tolerance='1H')
    y['pred'] = y[['light', 'moderate-vigorous', 'sedentary', 'sleep']].idxmax(axis=1)
    y["true"] = data["label"].resample('30S', origin="start").agg(lambda x: x.mode()[0] if not x.empty else None)
    y.to_csv(f"outputs/predictions/actinet/real_world/{pid}.csv")

for pid in tqdm(df_rw["pid"].unique()):
    get_rw_predictions(pid)

In [ ]:
df_wisdm = pd.read_csv("data/prepared/wisdm.csv", index_col="time", parse_dates=True)
df_wisdm["label"] = df_wisdm["label"].map({'teeth': 'light',
    'kicking': 'moderate-vigorous',
    'chips': 'sedentary',
    'jogging': 'moderate-vigorous',
    'typing': 'sedentary',
    'walking': 'moderate-vigorous',
    'clapping': 'light',
    'sandwich': 'sedentary',
    'sitting': 'sedentary',
    'folding': 'light',
    'standing': 'light',
    'soup': 'sedentary',
    'pasta': 'sedentary',
    'stairs': 'moderate-vigorous',
    'dribbling': 'moderate-vigorous',
    'writing': 'sedentary',
    'catch': 'light',
    'drinking': 'sedentary'})

In [ ]:
os.makedirs(f"outputs/predictions/actinet/wisdm", exist_ok=True)

def get_wisdm_predictions(pid):
    data = df_wisdm.loc[df_wisdm["pid"] == pid]
    y = clf.predict_from_frame(data, 30, sleep_tolerance='1H')
    y['pred'] = y[['light', 'moderate-vigorous', 'sedentary', 'sleep']].idxmax(axis=1)
    y["true"] = data["label"].resample('30S', origin="start").agg(lambda x: x.mode()[0] if not x.empty else None)
    y.to_csv(f"outputs/predictions/actinet/wisdm/{pid}.csv")

for pid in tqdm(df_wisdm["pid"].unique()):
    get_wisdm_predictions(pid)

In [ ]:
X = np.load("data/capture24/prepared/Walmsley2020/bbaa/X.npy")
y = np.load("data/capture24/prepared/Walmsley2020/bbaa/Y.npy")
T = np.load("data/capture24/prepared/Walmsley2020/bbaa/T.npy")
pids = np.load("data/capture24/prepared/Walmsley2020/bbaa/pid.npy")

In [ ]:
rf_params = {
    **{
        "winsec": WINSEC,
        "labels": list(np.unique(list(activity_labels))),
        "hmm_handle_sleep_transitions": False,
        "hmm_ignore_transition_gaps": False,
        "verbose": False
    },
    **MODEL_CONFIG[ANNOLABEL]["rf_params"]
}

clf_rf = RFActivityClassifier(**rf_params)
clf_rf.fit(X, y, pids, T)

In [ ]:
def extract_accelerometer_features(n_jobs):
    def process_file(filename, sample_hz):
        command = f'accProcess data/prepared/{filename}.csv --csvTimeFormat "yyyy-MM-dd HH:mm:ss.SSSSSSSSS" --csvTimeXYZTempColsIndex "0,1,2,3" -o "data/prepared/bbaa_features" --activityClassification False --deleteIntermediateFiles False --sampleRate {sample_hz} --timezone "UTC"'
        process = subprocess.run(command, shell=True, capture_output=True)

        return process

    Parallel(n_jobs=n_jobs)((delayed(process_file)(file, sample_hz) for file, sample_hz in [["real_world", 30],  ["wisdm", 30]]))

extract_accelerometer_features(2)

In [ ]:
rf_featues_cols = MODEL_CONFIG[ANNOLABEL]["rf_features"]

df_rw = pd.read_csv('data/prepared/real_world.csv', index_col="time", parse_dates=True)
df_rw["label"] = df_rw["label"].map({'sitting': 'sedentary', 
                                     'walking': 'moderate-vigorous', 
                                     'lying': 'sedentary', 
                                     'climbingdown': 'moderate-vigorous', 
                                     'climbingup': 'moderate-vigorous',
                                     'standing': 'light', 
                                     'jumping': 'moderate-vigorous', 
                                     'running': 'moderate-vigorous'})
df_rw_feats = pd.read_csv('data/prepared/bbaa_features/real_world-epoch.csv.gz', index_col="time", 
                          parse_dates=True, date_parser=custom_date_parser,
                          usecols=rf_featues_cols + ["time"])

In [ ]:
df_rw_feats["pid"] = df_rw["pid"].resample('30S').agg(lambda x: x.mode()[0] if not x.empty else None)
df_rw_feats["label"] = df_rw["label"].resample('30S').agg(lambda x: x.mode()[0] if not x.empty else None)

def get_rw_predictions(pid):
    data = df_rw_feats.loc[df_rw_feats["pid"] == pid]
    
    X = data[rf_featues_cols].values
    T = data.index.to_numpy()
    y = pd.DataFrame(clf_rf.predict(X, T, True, "1H"), index=T, columns=["pred"])
    y["true"] = data["label"]
    
    os.makedirs(f"outputs/predictions/bbaa/real_world", exist_ok=True)
    y.to_csv(f"outputs/predictions/bbaa/real_world/{pid}.csv")

for pid in tqdm(df_rw["pid"].unique()):
    get_rw_predictions(pid)

In [ ]:
df_wisdm = pd.read_csv('data/prepared/wisdm.csv', index_col="time", parse_dates=True)
df_wisdm["label"] = df_wisdm["label"].map({'teeth': 'light',
    'kicking': 'moderate-vigorous',
    'chips': 'sedentary',
    'jogging': 'moderate-vigorous',
    'typing': 'sedentary',
    'walking': 'moderate-vigorous',
    'clapping': 'light',
    'sandwich': 'sedentary',
    'sitting': 'sedentary',
    'folding': 'light',
    'standing': 'light',
    'soup': 'sedentary',
    'pasta': 'sedentary',
    'stairs': 'moderate-vigorous',
    'dribbling': 'moderate-vigorous',
    'writing': 'sedentary',
    'catch': 'light',
    'drinking': 'sedentary'})
df_wisdm_feats = pd.read_csv('data/prepared/bbaa_features/wisdm-epoch.csv.gz', index_col="time", parse_dates=True, 
                             date_parser=custom_date_parser, usecols=rf_featues_cols + ["time"])

In [ ]:
os.makedirs(f"outputs/predictions/bbaa/wisdm", exist_ok=True)

df_wisdm_feats["pid"] = df_wisdm["pid"].resample('30S').agg(lambda x: x.mode()[0] if not x.empty else None)
df_wisdm_feats["label"] = df_wisdm["label"].resample('30S').agg(lambda x: x.mode()[0] if not x.empty else None)

def get_wisdm_predictions(pid):
    data = df_wisdm_feats.loc[df_wisdm_feats["pid"] == pid]

    X = data[rf_featues_cols].values
    T = data.index.to_numpy()
    y = pd.DataFrame(clf_rf.predict(X, T, True, "1H"), index=T, columns=["pred"])
    y["true"] = data["label"]
    y.to_csv(f"outputs/predictions/bbaa/wisdm/{pid}.csv")

for pid in tqdm(df_wisdm["pid"].unique()):
    get_wisdm_predictions(pid)

Evaluation

In [ ]:
from pathlib import Path
from glob import glob
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score

models = {'bbaa': 'accelerometer', 'actinet': 'actinet'}
datasets = ['wisdm', 'real_world']
performance = {}
predictions = {}

for model in models.keys():
    performance[model] = {}
    predictions[model] = {}
    
    for dataset in datasets:
        pids = [Path(f).stem for f in glob(f'outputs/predictions/{model}/{dataset}/*.csv')]
        performance[model][dataset] = {}
        predictions[model][dataset] = {}

        for pid in pids:
            file_path = f'outputs/predictions/{model}/{dataset}/{pid}.csv'
            if os.path.exists(file_path):
                df = pd.read_csv(file_path, index_col=[0])
                df.dropna(inplace=True)
                y_true = df['true']
                y_pred = df['pred']
                
                performance[model][dataset][pid] = {
                    'accuracy': accuracy_score(y_true, y_pred),
                    'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
                    'macro_f1': f1_score(y_true, y_pred, average='macro'),
                    'cohen_kappa': cohen_kappa_score(y_true, y_pred)
                }

                predictions[model][dataset][pid] = {
                    'True': y_true.tolist(),
                    'Predicted': y_pred.tolist()
                }

performance_df = pd.DataFrame.from_dict({(models[i], j, k): performance[i][j][k]
                                         for i in performance.keys()
                                         for j in performance[i].keys()
                                         for k in performance[i][j].keys()},
                                        orient='index')

performance_summary = performance_df.groupby(level=[0, 1]).agg([np.mean, np.std])

predictions_df = pd.DataFrame.from_dict(
    {(models[i], j, k): predictions[i][j][k] for i in predictions.keys()
     for j in predictions[i].keys() for k in predictions[i][j].keys()},
    orient='index'
)

predictions_df.index.set_names(['Model', 'dataset', 'Participant'], inplace=True)
predictions_df.reset_index(inplace=True)

In [ ]:
performance_summary_pivot = performance_summary.stack(level=[0, 1]).unstack(level=[2, 3]).swaplevel(axis=0).sort_index(axis=0)

mean_values = performance_summary_pivot.xs('mean', level=1, axis=1)
std_values = performance_summary_pivot.xs('std', level=1, axis=1)

formatted_performance_summary = mean_values.applymap(lambda x: f'{x:.3f}') + \
    " ± " + std_values.applymap(lambda x: f'{x:.3f}')

formatted_performance_summary.to_csv('outputs/performance_summary.csv')
formatted_performance_summary

In [ ]:
generate_confusion_matrices(predictions_df[predictions_df["dataset"] == "wisdm"], list(activity_labels), 
                            save_path="outputs/actinet_vs_bbaa/conf_wisdm.png", fontsize=18)
generate_confusion_matrices(predictions_df[predictions_df["dataset"] == "real_world"], list(activity_labels), 
                            save_path="outputs/actinet_vs_bbaa/conf_rwe.png", fontsize=18)